# Core Multi-League Dataset — Limpieza

> **Ligas:** LaLiga · Premier League · Bundesliga&nbsp;&nbsp;◦&nbsp;&nbsp;**Temporadas:** 2014-15 → 2023-24&nbsp;&nbsp;◦&nbsp;&nbsp;**Fuente:** `core_multi_league_validated.parquet`

## Objetivos

- Convertir Date a datetime y transformar Div a League (nombres legibles).
- Generar match_id como clave compuesta y Season desde la fecha.
- Analizar cobertura de casas de apuestas y seleccionar B365 + Pinnacle.
- Imputar nulos residuales en cuotas de Pinnacle.
- Verificar integridad del dataset limpio (sanity check).

## Estructura del Notebook

| # | Sección | Objetivo |
|---|---------|----------|
| 0 | Entorno y configuración | Librerías y rutas del proyecto |
| 1 | Carga del dataset validado | Lectura del Parquet y estado inicial |
| 2 | Transformaciones | Date, League, equipos, Season y match_id |
| 3 | Gestión de nulos en cuotas | Cobertura por casa, selección B365 + Pinnacle |
| 4 | Validación post-limpieza | Sanity check del dataset limpio |
| 5 | Conclusiones | Síntesis de transformaciones y estado final |
| 6 | Exportación | Guardado en Parquet con esquema JSON |


##
---

## 0) Entorno y configuración

En esta sección se configuran las dependencias, librerías y parámetros globales necesarios para garantizar la reproducibilidad del análisis.

In [ ]:
from pathlib import Path
import pandas as pd
import sys
import json
from IPython.display import display

PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if any((p / m).exists() for m in {"config", "src", "data"})
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

CONFIG_ROOT = PROJECT_ROOT / "config"
with open(CONFIG_ROOT / "leagues.json") as f:
    ALL_LEAGUES = json.load(f)

PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed"
CORE_VALIDATED_PATH = PROCESSED_ROOT / "multi_league" / "core_multi_league_validated.parquet"
CORE_CLEAN_PATH = PROCESSED_ROOT / "multi_league" / "core_multi_league_clean.parquet"
METADATA_CLEAN_PATH = PROCESSED_ROOT / "multi_league" / "core_multi_league_clean_schema.json"

# Importación de funciones propias
from src.analysis import normalize_team_name
from src.utils import (
    get_shape_check,
    get_nulls_dict,
    get_duplicates_count,
    get_negatives_dict,
    get_invalid_categories,
)

---
##

## 1) Carga del core multi-league

Lectura del dataset unificado generado en el EDA comparativo y verificación del estado inicial antes de aplicar transformaciones.

In [ ]:
df = pd.read_parquet(CORE_VALIDATED_PATH)

dataset_divs = df["Div"].dropna().unique()
unknown = [d for d in dataset_divs if d not in ALL_LEAGUES]
if unknown:
    print(f"⚠ Ligas no reconocidas en config: {unknown}")

league_names = [ALL_LEAGUES[d] for d in dataset_divs if d in ALL_LEAGUES]

print(f"Dataset cargado: {CORE_VALIDATED_PATH.relative_to(PROJECT_ROOT)}")
print(f"  Filas: {len(df):,} | Columnas: {len(df.columns)}")
print(f"  Ligas: {len(league_names)} ({', '.join(league_names)})")


---
##

## 2) Limpieza básica

Se aplican las transformaciones básicas de saneamiento de datos, incluyendo la conversión de tipos, la validación de identificadores y la normalización de nombres.

### 2.1 Conversión de `Date` a datetime

Conversión de la columna `Date` a formato datetime. Si el Parquet ya la almacena como `datetime64`, se verifica directamente; en caso contrario se convierte desde texto.

In [ ]:
if not pd.api.types.is_datetime64_any_dtype(df["Date"]):
    df["Date"] = pd.to_datetime(df["Date"], format="mixed", dayfirst=True)
    print("Date: convertida de str a datetime\n")
else:
    print(f"Date: ya es {df['Date'].dtype}\n")

print("Cobertura temporal:")
for div in sorted(df["Div"].unique()):
    mask = df["Div"] == div
    min_d = df.loc[mask, "Date"].min().strftime("%d/%m/%Y")
    max_d = df.loc[mask, "Date"].max().strftime("%d/%m/%Y")
    name = ALL_LEAGUES.get(div, div)
    print(f"  • {name}: {min_d} → {max_d}")

assert df["Date"].isnull().sum() == 0, "Existen fechas nulas tras conversión"

### 2.2 Transformación de `Div` a `League`


Se aplica la función de normalización utilizada en los EDA individuales y se verifica que no existen colisiones.

In [ ]:
if "Div" in df.columns:
    df["League"] = df["Div"].map(ALL_LEAGUES)
    unmapped = df["League"].isnull().sum()
    if unmapped > 0:
        print(f"⚠ {unmapped} filas con Div no mapeado")
    else:
        print(f"Div → League: {df['League'].nunique()} ligas mapeadas")
        for div in sorted(df["Div"].unique()):
            print(f"  • {div} → {ALL_LEAGUES[div]} ({(df['Div'] == div).sum():,} partidos)")
    df = df.drop(columns=["Div"])
else:
    print(f"League ya existe ({df['League'].nunique()} ligas)")


### 2.3 Creación de `Season`

Creación de la columna `Season` a partir de `Date`. El corte en julio delimita cada temporada (agosto → mayo), evitando mezclar partidos de temporadas distintas que caen en el mismo año natural.

In [ ]:
df["Season"] = df["Date"].dt.to_period("Y-JUL").astype(str)

print(f"Season: {df['Season'].nunique()} temporadas\n")
for league in sorted(df["League"].unique()):
    mask = df["League"] == league
    seasons = sorted(df.loc[mask, "Season"].unique())
    print(f"  • {league}: {seasons[0]} → {seasons[-1]} ({len(seasons)} temporadas)")

### 2.4 Creación de `match_id`

Generación de clave compuesta única por partido. Formato: `{League}_{Season}_{HomeTeam}_{AwayTeam}` con nombres normalizados.

In [ ]:
df["match_id"] = (
    df["League"]
    + "_" + df["Season"]
    + "_" + df["HomeTeam"].apply(normalize_team_name).str.replace(" ", "_")
    + "_" + df["AwayTeam"].apply(normalize_team_name).str.replace(" ", "_")
)

n_unique = df["match_id"].nunique()

if n_unique == len(df):
    print(f"Se han creado {n_unique:,} claves únicas\n")
else:
    dups = df[df["match_id"].duplicated(keep=False)]
    print(f"  {len(df) - n_unique} duplicados:")
    display(dups[["match_id", "League", "Season", "HomeTeam", "AwayTeam", "FTR"]].head(10))

print("── Ejemplos de `match_id` por liga ───────────────────────────────\n")
for league in sorted(df["League"].unique()):
    sample = df.loc[df["League"] == league, "match_id"].iloc[0]
    print(f"   {league:<10}  →  {sample}")

print("\n──────────────────────────────────────────────────────────────────")

---
##

## 3) Gestión de nulos

Tratamiento de valores nulos en el dataset. Los nulos se concentran exclusivamente en variables de cuotas de apuestas, como se identificó en los EDA individuales.

### 3.1 Cobertura de casas de apuestas

Se analiza la cobertura de las casas de apuestas por temporada y de forma global para evaluar su estabilidad y decidir cuáles mantener en el dataset.

#### 3.1.1 Cobertura global

In [ ]:
bookmakers = {
    "B365": ["B365H", "B365D", "B365A"],
    "BW":   ["BWH", "BWD", "BWA"],
    "IW":   ["IWH", "IWD", "IWA"],
    "PS":   ["PSH", "PSD", "PSA"],
    "PS (cierre)": ["PSCH", "PSCD", "PSCA"],
    "VC":   ["VCH", "VCD", "VCA"],
    "WH":   ["WHH", "WHD", "WHA"],
}

print("Cobertura global por casa de apuestas:\n")
for bk, cols in bookmakers.items():
    available = [c for c in cols if c in df.columns]
    if available:
        coverage = (1 - df[available].isnull().mean().mean()) * 100
        nulls = df[available].isnull().sum().sum()
        print(f"  • {bk:12s}: {coverage:6.2f}%  ({nulls:,} nulos)")

#### 3.1.2 Cobertura por temporada


In [ ]:
print("Cobertura por casa y temporada (%):\n")

records = []

for bk, cols in bookmakers.items():
    available = [c for c in cols if c in df.columns]

    if available:
        for season in sorted(df["Season"].unique()):
            mask = df["Season"] == season
            cov = (1 - df.loc[mask, available].isnull().mean().mean()) * 100

            records.append({
                "Casa": bk,
                "Temporada": season,
                "Cobertura": round(cov, 2)
            })

cov_df = pd.DataFrame(records)
cov_pivot = cov_df.pivot_table(index="Casa", columns="Temporada", values="Cobertura")

display(cov_pivot.style.format("{:.2f}%"))

### 3.2 Selección de casas de referencia

Se seleccionan `B365` y `Pinnacle` como casas de referencia y se eliminan las restantes por redundancia.

In [ ]:
keep_prefixes = ["B365", "PS"]
drop_cols = [c for c in df.columns 
             if any(c.startswith(p) for p in ["IW", "BW", "VC", "WH"])]

cols_before = len(df.columns)
df = df.drop(columns=drop_cols)

print(f"Casas seleccionadas: B365 (retail) + PS (referencia)\n")
print(f"Columnas eliminadas ({len(drop_cols)}):")
print(f"  {', '.join(sorted(drop_cols))}")
print(f"\nDataset: {cols_before} → {len(df.columns)} columnas")

### 3.3 Imputación de nulos en las cuotas restantes 

Los nulos puntuales de `PS` se imputan con las cuotas equivalentes de`B365`. Los de cierre (`PSC`) se imputan con las de apertura (`PS`) del mismo partido.

In [ ]:
ps_map = {"PSH": "B365H", "PSD": "B365D", "PSA": "B365A"}
for ps_col, b365_col in ps_map.items():
    mask = df[ps_col].isnull()
    if mask.sum() > 0:
        df.loc[mask, ps_col] = df.loc[mask, b365_col]

psc_map = {"PSCH": "PSH", "PSCD": "PSD", "PSCA": "PSA"}
for close, opening in psc_map.items():
    mask = df[close].isnull()
    if mask.sum() > 0:
        df.loc[mask, close] = df.loc[mask, opening]

odds_cols = list(ps_map.keys()) + list(psc_map.keys()) + ["B365H", "B365D", "B365A"]
remaining = df[odds_cols].isnull().sum().sum()
print(f"Nulos en cuotas tras imputación: {remaining}")

### 3.4 Verificación de completitud 

Confirmación de que el dataset no contiene valores nulos tras el tratamiento.

In [ ]:
total_nulls = df.isnull().sum().sum()

if total_nulls == 0:
    print(f"Dataset completo: 0 nulos en {len(df.columns)} columnas × {len(df):,} filas")
else:
    null_cols = df.isnull().sum()
    null_cols = null_cols[null_cols > 0]
    print(f"⚠ {total_nulls} nulos restantes en {len(null_cols)} columnas:")
    for col, n in null_cols.items():
        print(f"  • {col}: {n} ({n/len(df)*100:.2f}%)")

---
##

## 4) Validación final post-limpieza

Sanity check rápido post-transformaciones. Las validaciones detalladas se realizaron en los EDA individuales y comparativo.

In [ ]:
valid, rows, cols = get_shape_check(df)
nulls = len(get_nulls_dict(df))
dups = get_duplicates_count(df, key_col="match_id")
negatives = len(get_negatives_dict(df))
invalid_cats = get_invalid_categories(df, "FTR", {"H", "D", "A"})
cats_text = "categorías válidas" if len(invalid_cats) == 0 else "⚠ Error en categorías"

print("── Verificación final ──────────────────────────────────────────────\n")
print(f"  {rows:,} filas × {cols} columnas")
print(f"  {nulls} nulos · {dups} duplicados · {cats_text}")
print("\n───────────────────────────────────────────────────────────────────")

---
##

## 5) Conclusiones de la fase de limpieza

Se aplicó un pipeline de limpieza unificado sobre el **core multi-league dataset** de **10.660 partidos × 43 variables**. Las principales transformaciones realizadas fueron:

### 5.1 Transformaciones aplicadas

**Date**  → conversión a `datetime`

**Div**  → creación de `League` (`bundesliga`, `laliga`, `premier`)

**Season**  → derivada de `Date` (temporada agosto → julio)

**match_id**  → clave compuesta `League_Season_Home_Away`

**Odds**  → selección de casas estables: `B365`, `PS`

> **Nota — imputación en Pinnacle (`PS`)**  
> Nulos en `PS_open` → imputados con `B365`  
> Nulos en `PS_close` → imputados con `PS_open`

### 5.2 Estado del dataset limpio

| Métrica | Valor |
|---------|-------|
| Partidos | 10.660 |
| Columnas | 33 |
| Nulos | 0 |
| Casas de apuestas | `B365`, `PS` (apertura), `PS` (cierre) |

El dataset resultante mantiene coherencia estructural y ausencia de valores faltantes, quedando preparado para la **fase de integración con el dataset de xG**.

---
##

## 6) Exportación del dataset limpio

Exportación del dataset limpio y metadatos del esquema para las fases posteriores del pipeline.

In [ ]:
metadata = {
    "num_columns": len(df.columns),
    "num_rows": len(df),
    "num_leagues": df["League"].nunique(),
    "leagues": sorted(df["League"].unique().tolist()),
    "columns": sorted(df.columns.tolist()),
    "dtypes": {col: str(df[col].dtype) for col in sorted(df.columns)},
    "matches_per_league": {lg: int((df["League"] == lg).sum()) for lg in sorted(df["League"].unique())}
}

df.to_parquet(CORE_CLEAN_PATH, index=False)

with open(METADATA_CLEAN_PATH, "w") as f:
    json.dump(metadata, f, indent=2)

print(f"Core clean exportado:")
print(f"  {len(df):,} filas × {len(df.columns)} columnas\n")
print(f"Archivos guardados:")
print(f"  · Dataset → {CORE_CLEAN_PATH.relative_to(PROJECT_ROOT)}")
print(f"  · Esquema → {METADATA_CLEAN_PATH.relative_to(PROJECT_ROOT)}")

---
##